In [53]:
import numpy as np
import cv2

In [54]:
# read depth image
depth_scale = 0.00012498664727900177
depth_img = cv2.imread('depth.png')
dpt = depth_img[:, :, 2] + depth_img[:, :, 1] * 256
dpt = dpt * depth_scale

# read seg image
seg = cv2.imread('seg.png')[...,0]  # 255: fore ground, 0: background

# read intrinsics and extrinsics
K = np.load('intrinsic.npy')
print(K)

[[415.69219382   0.         320.        ]
 [  0.         415.69219382 240.        ]
 [  0.           0.           1.        ]]


In [55]:
# task1: convert depth image to point cloud
def depth2pc(depth, seg, K):
    # ------------TODO---------------
    # compute point cloud from depth image
    # for-loop is not allowed!!
    # ------------TODO --------------

    # get the indices of the foreground pixels
    v, u = np.nonzero(seg)  # pixel coordinates are transposed compared to the numpy coordinatess
    alpha = K[0, 0]
    beta = K[1, 1]
    c_x = K[0, 2]
    c_y = K[1, 2]
    z = depth[v, u]
    x = z * (u - c_x) / alpha
    y = z * (v - c_y) / beta
    pc = np.concatenate((x.reshape(-1, 1), y.reshape(-1, 1), z.reshape(-1, 1)), axis=1)
    return pc

partial_pc = depth2pc(dpt, seg, K)

# For debug and submission
np.savetxt('../results/pc_from_depth.txt', partial_pc)

In [56]:
# task2: compute one-way chamfer distance to the complete shape
full_pc = np.loadtxt('aligned_full_pc.txt')

def random_sample(pc, num):
    permu = np.random.permutation(pc.shape[0])
    return pc[permu][:num]

partial_pc_sampled = random_sample(partial_pc, 2048)
full_pc_sampled = random_sample(full_pc, 2048)

# -----------TODO---------------
# implement one way chamfer distance
# -----------TODO---------------

partial_pc_sampled = partial_pc_sampled[:, np.newaxis, :]   # (2048, 1, 3)
full_pc_sampled = full_pc_sampled[np.newaxis, :, :] # (1, 2048, 3)
dis_mat = np.sqrt(np.sum((partial_pc_sampled - full_pc_sampled)**2, axis=2))   # (2048, 2048)
one_way_CD = np.sum(np.min(dis_mat, axis=1)) / 2048
one_way_CD = one_way_CD[None]
print('one way chamfer distance: ', one_way_CD)

# For submission
np.savetxt('../results/one_way_CD.txt', one_way_CD)

one way chamfer distance:  [0.01001395]


In [57]:
one_way_CD - 0.009976257336995639

array([3.76929078e-05])

In [58]:
# visualization
# import open3d

# visualized_pc = open3d.geometry.PointCloud() 
# visualized_pc.points = open3d.utility.Vector3dVector(partial_pc)
# open3d.visualization.draw_plotly([visualized_pc], mesh_show_wireframe=False)

In [59]:
# visualization
# import open3d

# visualized_pc = open3d.geometry.PointCloud() 
# visualized_pc.points = open3d.utility.Vector3dVector(full_pc)
# open3d.visualization.draw_plotly([visualized_pc], mesh_show_wireframe=False)